# AmEx Credit Card Product Assistant — Example Usage of `file_search` Library (v2)

This notebook demonstrates how to build a **realtime sales cue engine for AmEx credit cards** on top of the reusable `file_search` library.

**Use case:** During a customer call, a customer raises an objection or asks a question about an AmEx card → your existing system detects the objection and retrieves similar past objection-response pairs → this engine grounds the suggested response in authoritative AmEx product information (features, benefits, rewards, fees, APR, eligibility) to prevent hallucination.

**Why grounding matters:** Credit card products have regulated disclosures. Hallucinating an APR, a reward multiplier, or an annual fee has compliance implications, not just quality ones. This notebook demonstrates how to enforce strict product-to-chunk attribution via metadata.

This serves as a **reference pattern** for other teams building RAG use cases on the library.

## 1. Setup — import the library

In [ ]:
import os
from file_search import VectorStore, QueryEngine, split_query

## 2. One-time indexing with product metadata

Each file is indexed with metadata describing the product it represents. This metadata is:

1. **Prefixed into the chunk text** — so the LLM sees `[product: Platinum | short_name: Platinum | ...]` before every fact
2. **Stored structurally** — so code can filter retrieval using `filters={"short_name": "Platinum"}`

The `source` key is added automatically by the library — no need to include it manually.

In [ ]:
INDEX_PATH = "amex_card_index.parquet"

if not os.path.exists(INDEX_PATH):
    store = VectorStore()

    store.add_file(
        "amex_product_docs/platinum_card.pdf",
        metadata={
            "product": "The Platinum Card from American Express",
            "short_name": "Platinum",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "premium",
        }
    )
    store.add_file(
        "amex_product_docs/gold_card.pdf",
        metadata={
            "product": "American Express Gold Card",
            "short_name": "Gold",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/green_card.pdf",
        metadata={
            "product": "American Express Green Card",
            "short_name": "Green",
            "category": "Consumer Charge Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/blue_cash_preferred.pdf",
        metadata={
            "product": "Blue Cash Preferred Card from American Express",
            "short_name": "BCP",
            "category": "Consumer Credit Card",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/delta_skymiles.pdf",
        metadata={
            "product": "Delta SkyMiles Gold American Express Card",
            "short_name": "Delta Gold",
            "category": "Co-branded Airline",
            "annual_fee_tier": "mid",
        }
    )
    store.add_file(
        "amex_product_docs/membership_rewards_terms.md",
        metadata={
            "product": "Membership Rewards Program",
            "short_name": "MR Terms",
            "category": "Program Terms",
            "annual_fee_tier": "n/a",
        }
    )
    store.add_file(
        "amex_product_docs/fees_and_apr_disclosures.pdf",
        metadata={
            "product": "Fees & APR Disclosures",
            "short_name": "Disclosures",
            "category": "Regulatory Disclosure",
            "annual_fee_tier": "n/a",
        }
    )

    store.save(INDEX_PATH)
else:
    print(f"Index already exists at {INDEX_PATH}")

## 3. Load the index at startup

In your realtime service, load once at startup and reuse across all queries.

In [ ]:
product_store = VectorStore.load(INDEX_PATH)
print(f"Loaded {len(product_store.chunks)} chunks from {len({m['source'] for m in product_store.metadata})} sources")

# Inspect a sample chunk to confirm metadata prefix is in place
print("\nSample chunk:")
print(product_store.chunks[0][:400])

## 4. Configure the QueryEngine for AmEx card cue

The system message tells the LLM how to use the metadata prefix for strict product attribution. This is critical for credit card products to prevent cross-contamination of features between cards.

In [ ]:
AMEX_CARD_SYSTEM_MESSAGE = (
    "You are a real-time sales assistant helping representatives respond to customer "
    "questions and objections about American Express credit cards. Suggest a concise "
    "response (2-3 sentences) the rep can use.\n\n"
    "The CONTEXT below contains authoritative product facts — the official terms and "
    "disclosures. Do not contradict or supplement them.\n\n"
    "CRITICAL RULES:\n"
    "1. Every product fact below is prefixed with a metadata tag in the format "
    "[product: <card name> | short_name: <nickname> | category: <type> | ... | source: <file>]. "
    "You MUST attribute each feature, fee, benefit, or term to the specific card "
    "identified by its 'product' value. NEVER describe a feature from one card as "
    "belonging to another.\n"
    "2. If the customer is asking about a specific card, use ONLY facts whose "
    "metadata matches that card. Ignore facts about other cards unless the "
    "customer explicitly asked for a comparison.\n"
    "3. When comparing cards, clearly label which feature belongs to which card. "
    "Example: 'The Platinum Card offers X, while the Gold Card offers Y.'\n"
    "4. All product details (fees, APRs, reward rates, benefits) MUST come from the "
    "CONTEXT. NEVER invent values — incorrect disclosures have compliance "
    "implications.\n"
    "5. If CONTEXT for the specific card asked about is missing, instruct "
    "the rep to confirm and follow up rather than guess.\n"
    "6. Output ONLY the suggested response — no preamble."
)

engine = QueryEngine(
    product_store,
    system_message=AMEX_CARD_SYSTEM_MESSAGE,
)

## 5. Demo — large, messy customer query

In practice the input is messy — a full call transcript, a pasted email, a batch of utterances, a multi-topic question. You assemble whatever prompt you want from your own data, then let the library split it into sub-queries, retrieve per sub-query, RRF-merge across sub-queries, and synthesize a grounded answer.

In [ ]:
# A deliberately messy, multi-topic customer input.
# In production this could be a call transcript, pasted email,
# concatenated chat history, etc. — whatever you've got.
messy_query = """
Hey so I was looking at the Platinum Card, $695 seems steep honestly,
what do I actually get for it, also my wife spends $1500/month on
groceries so is Gold or Blue Cash Preferred better for her, and does
the Delta SkyMiles card cover lounge access at DTW? One more thing,
what's the foreign transaction fee on Green? Sorry for the long message.
"""

# 1. Split the messy input into focused word-window sub-queries.
sub_queries = split_query(messy_query)
print(f"Split into {len(sub_queries)} sub-queries:")
for q in sub_queries:
    print(f"  - {q!r}")

# 2. Hybrid-retrieve per sub-query, RRF-merge across sub-queries.
hits = product_store.retrieve(sub_queries, top_k=10, final_k=5)
print(f"\nRetrieved {len(hits)} chunks (RRF-merged):")
for h in hits:
    print(f"  [score={h['score']:.4f}] {h['source']} "
          f"— {h['metadata'].get('short_name')}")

# 3. Synthesize a grounded answer using the retrieved chunks.
result = engine.synthesize(messy_query, hits)
print("\nANSWER:\n")
print(result["answer"])

## 6. Tuning knobs

| Knob | Default | Notes |
|---|---|---|
| `top_k` | `10` | Candidates per sub-query |
| `final_k` | `5` | Chunks returned after RRF merge |
| `rerank` | `False` | LLM rerank on single-query path only |
| `filters` | `None` | Metadata filter; AND across keys, OR within list values |
| `SUBQUERY_SIZE` | `60` | Words per sub-query (module constant in `file_search.py`) |
| `SUBQUERY_OVERLAP` | `30` | Word overlap between sub-queries |

All network calls (embeddings and chat completions) are wrapped in a universal burst-retry: within a burst, 5 rapid attempts (~0.2 s apart); between bursts, exponential backoff (2 / 4 / 8 / 16 s); up to 5 bursts total (25 max attempts per call). Retries trigger on **any** exception — transient network errors, rate limits, unexpected API responses all get the same treatment.